# Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

E0000 00:00:1779411314.976454 2129815 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1779411314.976493 2129815 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1779411314.976498 2129815 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1779411314.976503 2129815 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1779411314.976506 2129815 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


In [2]:
# .env file contains the following variables:
# OPENAI_API_KEY
# CHROMA_OPENAI_API_KEY
# TAVILY_API_KEY

# load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### VectorDB Instance

In [3]:
# instantiate chromadb client
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [4]:
# open_ai embedding function
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    model_name="text-embedding-3-small"
)

In [5]:
collection = chroma_client.create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [6]:
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

### Semantic Search Demonstration

In [7]:
query = "Are there games that involve racing?"

results = collection.query(
    query_texts=[query],
    n_results=3,
)

documents = results["documents"][0]
metadatas = results["metadatas"][0]
distances = results["distances"][0]

for i in range(len(documents)):
    document = documents[i]
    metadata = metadatas[i]
    distance = distances[i]

    print(f"Result #{i + 1}")
    print(f"Name: {metadata['Name']}")
    print(f"Platform: {metadata['Platform']}")
    print(f"Genre: {metadata['Genre']}")
    print(f"Year: {metadata['YearOfRelease']}")
    print(f"Distance: {distance:.4f}")
    print(f"Document: {document}\n")

Result #1
Name: Gran Turismo 5
Platform: PlayStation 3
Genre: Racing
Year: 2010
Distance: 0.5526
Document: [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.

Result #2
Name: Gran Turismo
Platform: PlayStation 1
Genre: Racing
Year: 1997
Distance: 0.5641
Document: [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.

Result #3
Name: Mario Kart 8 Deluxe
Platform: Nintendo Switch
Genre: Racing
Year: 2017
Distance: 0.6859
Document: [Nintendo Switch] Mario Kart 8 Deluxe (2017) - An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.

